# Pengayaan Python — opsional, tidak wajib

**Guru Tamu Kecerdasan Artifisial · SMKN Bali Mandara · 24–25 September 2026**  
Narasumber: Dewa Ketut Satriawan Suditresnaya — Senior Software Engineer & AI Researcher, PT. Bima Sakti Alterra

---

Notebook ini **bukan bagian dari jalur praktik utama**. Seluruh proyek PilahPintar dapat
diselesaikan tanpa menyentuh satu baris pun kode di sini.

Notebook ini untuk kamu yang sudah selesai lebih cepat dan ingin tahu seperti apa
pekerjaan yang sama dikerjakan dengan Python. Ada tiga bagian pendek:

1. Membaca berkas CSV dan menghitung ringkasannya
2. Memisahkan data latih dan data uji — konsep yang sama seperti pada Modul Siswa 2.3
3. Melatih model sederhana dan melihat hasilnya pada data uji

**Waktu:** sekitar 20 menit.

## Yang perlu ada sebelum mulai

| Kebutuhan | Keterangan |
|---|---|
| Python | 3.9 atau lebih baru |
| Paket | hanya pustaka bawaan Python (`csv`, `random`, `statistics`) |
| Berkas data | `konsumsi-listrik-sekolah-sintetis.csv`, satu folder dengan notebook ini |

Tidak perlu memasang pandas, scikit-learn, atau TensorFlow. Pilihan ini disengaja:
pemasangan paket sering gagal di jaringan sekolah dan menghabiskan waktu yang seharusnya
dipakai belajar.

## Tentang datanya

> **Data ini sintetis.** Angka-angkanya dibuat dengan program untuk latihan, **bukan hasil
> pengukuran nyata** di SMKN Bali Mandara atau sekolah mana pun. Jangan memakainya sebagai
> bukti performa, bahan laporan konsumsi listrik, atau dasar pengambilan keputusan.

Kamus datanya ada pada berkas `KAMUS-DATA.md` di folder `data-contoh`.


## Bagian 1 — Membaca CSV dan menghitung ringkasan

Jalankan sel di bawah. Kalau muncul `FileNotFoundError`, berarti berkas CSV tidak berada
di folder yang sama dengan notebook ini.


In [ ]:
import csv, os

BERKAS = 'konsumsi-listrik-sekolah-sintetis.csv'

if not os.path.exists(BERKAS):
    raise FileNotFoundError(
        f"Berkas '{BERKAS}' tidak ditemukan di {os.getcwd()}. "
        'Letakkan notebook dan berkas CSV dalam satu folder.')

with open(BERKAS, encoding='utf-8') as f:
    baris = list(csv.DictReader(f))

print('Jumlah baris :', len(baris))
print('Nama kolom   :', list(baris[0].keys()))
print()
print('Tiga baris pertama:')
for b in baris[:3]:
    print(' ', b)


Sekarang hitung total pemakaian per jenis ruang.

Perhatikan `float(b['kwh'])`: nilai dari CSV selalu terbaca sebagai teks, jadi harus
diubah dulu menjadi angka sebelum dijumlahkan. Ini sumber kesalahan yang paling sering
terjadi saat pertama kali mengolah CSV.


In [ ]:
total = {}
jumlah = {}

for b in baris:
    jenis = b['jenis_ruang']
    total[jenis] = total.get(jenis, 0.0) + float(b['kwh'])
    jumlah[jenis] = jumlah.get(jenis, 0) + 1

print(f"{'jenis ruang':<12} {'total kWh':>10} {'rata-rata':>10} {'baris':>7}")
print('-' * 42)
for jenis in sorted(total, key=lambda j: -total[j]):
    print(f'{jenis:<12} {total[jenis]:>10.2f} '
          f'{total[jenis]/jumlah[jenis]:>10.2f} {jumlah[jenis]:>7}')


**Checkpoint 1.** Apakah keluaran di atas menunjukkan tiga jenis ruang: `lab`, `kelas`,
dan `umum`? Kalau ya, lanjutkan.

**Latihan.** Ubah sel di atas agar mengelompokkan berdasarkan kolom `hari`, bukan
`jenis_ruang`. Hari apa yang totalnya paling rendah? Apakah hasilnya masuk akal?


## Bagian 2 — Memisahkan data latih dan data uji

Ini konsep yang sama dengan yang kamu pakai pada proyek PilahPintar: sebagian data
disembunyikan selama pelatihan, lalu dipakai untuk menilai.

Ada satu hal penting pada data ini. Setiap ruang muncul di banyak tanggal. Kalau kita
mengacak seluruh baris begitu saja, baris dari ruang yang sama bisa jatuh di data latih
sekaligus di data uji — dan itulah bentuk **kebocoran data** pada data tabel.

Karena itu kita membagi **berdasarkan tanggal**: lima hari terakhir menjadi data uji,
sisanya data latih. Model dilatih pada masa lalu, diuji pada masa yang belum ia lihat.


In [ ]:
tanggal_unik = sorted({b['tanggal'] for b in baris})
tanggal_uji = set(tanggal_unik[-5:])

latih = [b for b in baris if b['tanggal'] not in tanggal_uji]
uji   = [b for b in baris if b['tanggal'] in tanggal_uji]

print('Total tanggal   :', len(tanggal_unik))
print('Tanggal untuk uji:', sorted(tanggal_uji))
print()
print(f'Data latih : {len(latih):>3} baris')
print(f'Data uji   : {len(uji):>3} baris')
print(f'Porsi uji  : {len(uji)/len(baris):.0%}')

bocor = {b['tanggal'] for b in latih} & tanggal_uji
print()
print('Tanggal yang muncul di kedua kelompok:', bocor if bocor else 'tidak ada — aman')


**Checkpoint 2.** Baris terakhir harus mencetak `tidak ada — aman`. Kalau tidak,
pembagiannya salah dan seluruh hasil pengujian berikutnya tidak berarti.

Pemeriksaan semacam ini layak kamu tulis setiap kali membagi data — jauh lebih murah
daripada menemukan kebocoran setelah hasilnya terlanjur dilaporkan.


## Bagian 3 — Model sederhana dan pengujiannya

Model yang kita buat sesederhana mungkin, supaya seluruh isinya bisa kamu baca:
**untuk setiap jenis ruang, tebak nilai rata-rata kWh dari data latih.**

Ini memang model yang naif. Tetapi ia memenuhi definisi model pada Modul Siswa 2.1:
hasil belajar dari data latih, yang tersimpan dan bisa dipakai untuk menebak contoh baru.

Model seperti ini disebut **baseline**. Gunanya sebagai pembanding: model yang lebih rumit
baru layak dipakai kalau hasilnya mengalahkan baseline.


In [ ]:
# TRAINING — model belajar dari data latih saja
jumlah_latih, total_latih = {}, {}
for b in latih:
    j = b['jenis_ruang']
    total_latih[j] = total_latih.get(j, 0.0) + float(b['kwh'])
    jumlah_latih[j] = jumlah_latih.get(j, 0) + 1

model = {j: total_latih[j] / jumlah_latih[j] for j in total_latih}

print('Isi model (rata-rata kWh per jenis ruang, dari data latih):')
for j, v in sorted(model.items()):
    print(f'  {j:<8} {v:>7.2f} kWh')


In [ ]:
# INFERENCE — model dipakai pada data uji yang belum pernah dilihatnya

def tebak(b):
    return model.get(b['jenis_ruang'], sum(model.values()) / len(model))

selisih = [abs(tebak(b) - float(b['kwh'])) for b in uji]
mae_uji = sum(selisih) / len(selisih)

selisih_latih = [abs(tebak(b) - float(b['kwh'])) for b in latih]
mae_latih = sum(selisih_latih) / len(selisih_latih)

print(f'Rata-rata selisih pada DATA LATIH : {mae_latih:.2f} kWh')
print(f'Rata-rata selisih pada DATA UJI   : {mae_uji:.2f} kWh')
print()
print(f'Diuji pada {len(uji)} baris dari {len(tanggal_uji)} tanggal.')


Angka di atas disebut **MAE** (*mean absolute error*) — rata-rata besarnya kesalahan
tebakan, dalam satuan yang sama dengan datanya, yaitu kWh.

Perhatikan cara melaporkannya: selalu sebutkan **berapa banyak data uji** yang dipakai.
Aturan yang sama berlaku untuk proyek PilahPintar — tulis "9 dari 12 benda uji benar",
bukan "akurasi tinggi".


In [ ]:
# Lima tebakan pertama pada data uji, supaya angkanya tidak terasa abstrak
print(f"{'ruang':<22}{'jenis':<8}{'sebenarnya':>11}{'tebakan':>9}{'selisih':>9}")
print('-' * 59)
for b in uji[:5]:
    asli, t = float(b['kwh']), tebak(b)
    print(f"{b['ruang']:<22}{b['jenis_ruang']:<8}{asli:>11.2f}{t:>9.2f}{abs(t-asli):>9.2f}")


**Checkpoint 3.** Apakah MAE pada data uji lebih besar daripada pada data latih?

Pada percobaan penyusun notebook ini, MAE data latih sekitar 1,7 kWh dan MAE data uji
sekitar 2,3 kWh. Angkamu bisa sedikit berbeda kalau berkas CSV-nya diganti, tetapi polanya
sama: hasil pada data uji lebih buruk. Itu wajar — data uji berisi tanggal yang belum
pernah dilihat model.

Selisihnya di sini masih tergolong sedang, justru karena modelnya terlalu sederhana untuk
bisa menghafal. Model yang jauh lebih rumit dengan data sesedikit ini akan menunjukkan
selisih yang jauh lebih lebar — itulah **overfitting** yang dibahas di Modul Siswa 2.4.

## Tantangan lanjutan

1. Buat model kedua yang menebak berdasarkan kolom `ruang`, bukan `jenis_ruang`.
   Apakah MAE-nya membaik? Apa risikonya kalau ada ruang yang hanya muncul sekali?
2. Tambahkan pengaruh `jam_pakai` ke dalam tebakan, misalnya
   `rata-rata kWh per jam × jam_pakai`. Bandingkan MAE-nya dengan baseline.
3. Ubah pembagian data menjadi acak (bukan berdasarkan tanggal), lalu bandingkan MAE-nya.
   Apakah hasilnya terlihat lebih baik? Apakah hasil yang terlihat lebih baik itu
   benar-benar lebih baik, atau hanya akibat pembagian yang lebih longgar?

## Kalau kamu ingin melanjutkan sendiri

Langkah berikutnya yang wajar adalah `pandas` untuk mengolah tabel dan `scikit-learn`
untuk model. Keduanya perlu dipasang, dan pemasangannya sebaiknya dilakukan di rumah
atau di jaringan yang tidak dibatasi:

```
python -m venv .venv
# Linux atau macOS:  source .venv/bin/activate
# Windows:           .venv\Scripts\activate
pip install pandas scikit-learn
```

Gunakan lingkungan virtual (`venv`) seperti di atas, jangan memasang ke Python sistem.

---

*Notebook ini pengayaan opsional dan tidak dinilai. Data di dalamnya sintetis.*
